# Phase 1 — Parse the exam corpus: audit → page map → `parse_exam.py`

Runs the exam-parsing pipeline end to end: `eval/audit_exams.py` (per-year format
diagnostics), `eval/build_page_map.py` (printed-page → PDF-page map, anchored on the
Bolognia table of contents), and `eval/parse_exam.py` (the real output: 949 questions,
bucketed and resolved to `chunks.jsonl` chunk IDs). See `derma_guide_plan.md` Phase 1
for the full write-up these choices are based on — three reference-file structures,
two numbering conventions, two different broken-glyph bugs, one cross-version
alignment table (2023), and a crop-box-intersection fix for image-page association.

No GPU needed — this is pure PDF parsing (pymupdf), CPU is fine on Colab's free tier
or locally.

**Files to upload** (all copyrighted / not redistributed — this notebook is written
to be reproducible by someone with their own copy of these on their own Drive, not by
an anonymous third party with no access to either corpus) into
`/content/drive/MyDrive/derma_force_eval/`:
```
Bolognia Dermatology 5th 2024.pdf
data/exams/{2021,2022,2023,2024_05,2024_09,2025,2026}/{questions,answers,references}.pdf
pipeline/data/chunks/chunks.jsonl        <- from the pipeline's own run (Phase 2)
eval/
  audit_exams.py  build_page_map.py  parse_exam.py
```
The layout under `/content/derma_force/` mirrors the repo root exactly so the scripts'
relative-path assumptions (`ROOT = Path(__file__).resolve().parent.parent`, etc.) just
work unmodified.

## 1. Setup

In [ ]:
!pip install -q pymupdf

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

EVAL_DRIVE_DIR = '/content/drive/MyDrive/derma_force_eval'   # see the file layout above

import os
REPO_ROOT = '/content/derma_force'
if not os.path.exists(REPO_ROOT):
    os.symlink(EVAL_DRIVE_DIR, REPO_ROOT)
print("repo root:", REPO_ROOT, "->", os.readlink(REPO_ROOT) if os.path.islink(REPO_ROOT) else "(real dir)")
!ls {REPO_ROOT}
!ls {REPO_ROOT}/data/exams

## 2. Audit — per-year format diagnostics

Confirms the format variation documented in Phase 1 before trusting the real parser
against it: numbering style, glyph-bug detection, answer-key schema, reference
chapter-token counts. Red flags to check in the printed table: `Qs` should be 150 (or
100 for the two known 100-question sittings), `schema=ALIGN-TBL` should appear for
2023 only.

In [ ]:
%cd {REPO_ROOT}/eval
!python3 audit_exams.py

## 3. Page map — printed-book-page → Bolognia-PDF-page

One-time, ~30s. Anchors on ~160 chapter-start pages from the PDF's own table of
contents rather than assuming a constant offset (verified wrong — the true offset
grows from ~40 near the front of this 2-volume book to ~490 by the back). Writes
`data/bolognia_page_map.json`, which `parse_exam.py` needs for every reference whose
format gives a page but no chapter (5 of the 7 exam years).

In [ ]:
!python3 build_page_map.py
!ls -la {REPO_ROOT}/data/bolognia_page_map.json

## 4. Parse — all seven years

Per-file numbering-pattern detection (`N.` vs. the reversed `.N`), per-file glyph-bug
detection, the 2023 cross-version alignment-table parser, crop-box-intersected image
association, and page/chapter resolution against `chunks.jsonl`. Writes
`eval/data/exam_parsed/{year}.jsonl`, `all.jsonl`, and `summary.json`.

Expect **949 total** across the printed per-year table (`text=` + `image=` + `lever=`
+ `unresolved=` should sum to each year's `n=`); see `derma_guide_plan.md` Phase 1e
for the exact reference table this should reproduce.

In [ ]:
!python3 parse_exam.py

## 5. Pull the parsed output back to Drive, so it survives the runtime disconnecting

In [ ]:
import shutil, os
os.makedirs(f"{EVAL_DRIVE_DIR}/eval/data/exam_parsed", exist_ok=True)
os.makedirs(f"{EVAL_DRIVE_DIR}/data", exist_ok=True)
for fname in ["exam_parsed/all.jsonl", "exam_parsed/summary.json"] + [
    f"exam_parsed/{y}.jsonl" for y in
    ["2021", "2022", "2023", "2024_05", "2024_09", "2025", "2026"]
]:
    src = f"{REPO_ROOT}/eval/data/{fname}"
    if os.path.exists(src):
        shutil.copy(src, f"{EVAL_DRIVE_DIR}/eval/data/{fname}")
        print("saved ->", f"{EVAL_DRIVE_DIR}/eval/data/{fname}")
shutil.copy(f"{REPO_ROOT}/data/bolognia_page_map.json", f"{EVAL_DRIVE_DIR}/data/bolognia_page_map.json")
print("saved -> bolognia_page_map.json")

## Next steps (Phase 3)

With `exam_parsed/all.jsonl` and `bolognia_page_map.json` on Drive:
- Build the FAISS indices (`eval/build_index.py`) and run the retrieval grid
  (`eval/eval_retrieval.py`) — `notebooks/02_retrieval_eval.ipynb`.
- The develop years (2021–2024) feed retrieval tuning; 2025+2026 stay held out for
  Phase 5's 4-arm generation eval (`notebooks/04_generation_eval.ipynb`).